In [11]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

In [12]:
FP_RADIUS = 2      # neighborhood size
FP_BITS = 1024     # fingerprint length

In [13]:
#columns with INDEX, SMILES, ACTIVE
train_df = pd.read_csv('../Datasets/training_smiles.csv')
#columns with INDEX and SMILES
test_df  = pd.read_csv('../Datasets/test_smiles.csv')

y_train = train_df["ACTIVE"].to_numpy()

#show the first few rows of the table
train_df.head(), test_df.head()

(   INDEX                                             SMILES  ACTIVE
 0      1           O=C(Nc1ccc2c(c1)OCCO2)C1CCN(c2ncccn2)CC1     0.0
 1      2  COCCCN1C(=O)C2C(C(=O)Nc3cccc(Cl)c3)C3C=CC2(O3)...     0.0
 2      3                   CCSc1ncc(Cl)c(C(=O)Nc2ccccc2C)n1     0.0
 3      4  COc1ccc2cc(/C=N/NC(=O)CN(c3ccccc3C)S(=O)(=O)c3...     0.0
 4      5  CCCC(=O)Nc1nc2ccc(NC(=O)c3c(F)c(F)c(OC)c(F)c3F...     0.0,
     INDEX                                           SMILES
 0  202896                    O=C(N/N=C/c1ccc(Br)s1)c1ccco1
 1  202897  Cc1nc(SCC(=O)Nc2c(C)n(C)n(-c3ccccc3)c2=O)n[nH]1
 2  202898                       O=C(NC(=S)NCc1ccccc1)C1CC1
 3  202899                   C/C=C/C(=O)NCCc1ccc(OC)c(OC)c1
 4  202900    Cc1cc(C)c(OCC(=O)N/N=C/c2ccc3c(c2)OCO3)c(C)c1)

In [14]:
# add a Mol column (can reuse for descriptors & fingerprints)
train_df["mol"] = train_df["SMILES"].apply(Chem.MolFromSmiles)
test_df["mol"]  = test_df["SMILES"].apply(Chem.MolFromSmiles)

[19:06:16] WARNING: not removing hydrogen atom without neighbors
[19:06:26] WARNING: not removing hydrogen atom without neighbors


In [15]:
def compute_morgan_fp(mol, fpgen, n_bits=FP_BITS):
    """
    Return a numpy array (length = n_bits) with the Morgan fingerprint
    for a single RDKit Mol. If mol is None, returns all zeros.
    """
    if mol is None:
        return np.zeros(n_bits, dtype=np.int8)
    
    fp = fpgen.GetFingerprint(mol)      # RDKit ExplicitBitVect
    arr = np.array(fp, dtype=np.int8)   # convert to 0/1 numpy array
    return arr

In [16]:
fpgen = AllChem.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

In [ ]:
# TRAIN fingerprints
fps_train = []

for mol in train_df["mol"]:
    fp_vec = compute_morgan_fp(mol, fpgen, n_bits=FP_BITS)
    fps_train.append(fp_vec)

X_train_fp = np.vstack(fps_train)
print("X_train_fp shape:", X_train_fp.shape)  


# TEST fingerprints
fps_test = []

for mol in test_df["mol"]:
    fp_vec = compute_morgan_fp(mol, fpgen, n_bits=FP_BITS)
    fps_test.append(fp_vec)

X_test_fp = np.vstack(fps_test)
print("X_test_fp shape:", X_test_fp.shape)  



X_train_fp shape: (202895, 1024)
X_test_fp shape: (67631, 1024)


In [21]:
y_train.shape


(202895,)

In [23]:
print(X_train_fp[0], len(X_train_fp[0]))

[0 0 0 ... 0 0 0] 1024


In [18]:
np.save("X_train_morgan_fp.npy", X_train_fp)
np.save("X_test_morgan_fp.npy", X_test_fp)
np.save("y_train.npy", y_train)